# Embedding comparison: raw pixels vs MAE

Does a learned MAE representation give a **more clusterable** patch space than raw
pixels?  Every arm goes through the identical pipeline --

    features -> UMAP(3D) -> KMeans

-- and is scored with the same three diagnostics, so the only thing that varies is
the representation.

## The diagnostics

**1. Stability (the decisive one).**  Fit KMeans on two *disjoint* subsamples, use
each to label a common held-out set, and take the ARI between the two labellings.
A partition that survives redrawing the data is real; one that doesn't is an
arbitrary tessellation of a continuum.  Read the ARI-vs-k curve: the k where it
falls away is the finest structure the representation actually supports.

A `same-sample, different-seed` control runs alongside it, so KMeans initialisation
noise can be told apart from genuine sampling instability.

**2. Neighbour consistency / geography** (`metrics.embedding_eval`).  Are a patch's
nearest neighbours physically more similar than random (ratio < 1, good), and are
they merely *geographically* co-located (ratio < 1, bad -- the manifold is a map)?

**3. Position confound.**  MAE adds a learned positional embedding before the
encoder, so two identical patches at different grid positions embed differently.
Because cutouts are gradb2-weighted-sampled, cutout *centres* are systematically
frontal -- position correlates with front-ness.  If a patch's neighbours share its
within-cutout grid position, apparent structure may be position, not physics.

## How to read the result

MAE wins if its ARI curve stays high to larger k **and** its neighbour-consistency
ratio is lower **and** its geographic + position ratios are no worse.  If the ARI
curves lie on top of each other, the representation is not the binding constraint.

In [ ]:
import gc, pickle, time
from contextlib import contextmanager

import torch                       # import torch BEFORE cuML: CUDA/numba init order
import cuml
from cuml.cluster import KMeans
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.metrics import adjusted_rand_score

from nemi import SingleNemi
import llc_cutout_dataloader.cutouts_dataset as cutouts_dataset
import dl_embedding.mae as mae
import metrics.embedding_eval as ev

try:
    import cupy as cp
except Exception:
    cp = None

In [ ]:
source = cutouts_dataset.CutoutDataSource(bucket="dbof", folder="cutouts_dataset_v2",
                                          run_id="2_02",
                                          dataset_name="cutout_dataset.zarr",
                                          s3_endpoint="https://s3-west.nrp-nautilus.io")
source.print_available_channels()

In [ ]:
data_channels = ['Eta','Salt','Theta','W','gradb2','turner_angle','strain_mag',
                 'divergence','relative_vorticity','oceQnet','ekman_pumping','wind_stress_curl']

dataset = cutouts_dataset.CutoutDataset.from_source(
    data_channels=data_channels, source=source, subset=False,
    subsample_per_chunk=64, num_sample_chunks=1, n_workers=32)

## Configuration

`SWEEP_N` subsamples the patches once; **every arm is scored on that same index set**,
so the comparison is paired.  MAE still trains on all cutouts -- only the scoring is
subsampled.

`MAE_EPOCHS` dominates the runtime; everything else is minutes.

In [ ]:
PATCH_SIZE   = 8
UMAP_PARAMS  = {"n_components": 3, "n_neighbors": 80, "min_dist": 0.05}
K_VALUES     = [5, 20, 50, 150, 350, 1000]
SWEEP_N      = 400_000       # patches scored per arm (None = all)
N_SUB        = 60_000        # per stability subsample; two disjoint draws + an eval set
N_EVAL       = 60_000
KNN_N        = 60_000        # patches used for the neighbour metrics (kNN is O(N^2)-ish)
KNN_K        = 10
MAE_EPOCHS   = 40
SEED         = 0

# Arms to run.  raw_anomaly is a free control: raw pixels with the per-patch,
# per-channel mean removed.  If it alone fixes separability, the problem was the
# patch level and MAE is not needed to solve it.
ARMS = {"raw": True, "raw_anomaly": True, "mae": True, "mae_normpix": True}

rng = np.random.default_rng(SEED)

def free_gpu():
    gc.collect()
    if cp is not None:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()

## Features

In [ ]:
patches = dataset.get_patches(patch_size=PATCH_SIZE)      # (Np, C*p*p), log-grads + z-score
N_PATCHES = len(patches)
print("patches", patches.shape)

# one shared index set for every arm
sweep_idx = (np.sort(rng.choice(N_PATCHES, SWEEP_N, replace=False))
             if SWEEP_N and SWEEP_N < N_PATCHES else np.arange(N_PATCHES))
print("scoring on", len(sweep_idx), "patches")

# per-patch descriptors, coords and within-cutout grid position, aligned to get_patches order
desc, lon, lat, chan_names = ev.patch_descriptors(dataset, PATCH_SIZE)
grid  = dataset.X.shape[2] // PATCH_SIZE
ppi   = grid * grid                                        # patches per cutout
pos   = np.arange(N_PATCHES) % ppi                         # row-major index within its cutout
pos_rc = np.stack(np.divmod(pos, grid), axis=1).astype("float32")   # (Np, 2) grid row/col
print(f"grid {grid}x{grid} = {ppi} patches per cutout")

### MAE arms

`mae_normpix` uses the original MAE paper's per-patch **normalized** reconstruction
target, so the loss stops rewarding getting the patch *level* right and pushes the
encoder toward texture.  That matters here because the level is regional
oceanography -- the continuum that has been dominating the manifold.

`mae.PatchMAE` is swapped for the subclass only inside the `norm_pix_target()`
block, so `src/dl_embedding/mae.py` is untouched.

In [ ]:
class NormPixMAE(mae.PatchMAE):
    """PatchMAE with the per-patch normalized reconstruction target (He et al. 2021)."""

    def forward(self, imgs, mask_ratio):
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        target = self.patchify(imgs)
        mean = target.mean(dim=-1, keepdim=True)
        var = target.var(dim=-1, keepdim=True)
        target = (target - mean) / (var + 1.0e-6) ** 0.5
        loss = ((pred - target) ** 2).mean(dim=-1)
        return (loss * mask).sum() / mask.sum()


@contextmanager
def norm_pix_target():
    original, mae.PatchMAE = mae.PatchMAE, NormPixMAE
    try:
        yield
    finally:
        mae.PatchMAE = original


def train_mae(images, epochs=MAE_EPOCHS, seed=SEED):
    """Train an MAE on an 80/20 cutout split and embed every cutout to patch tokens."""
    perm = np.random.default_rng(seed).permutation(len(images))
    n_val = max(1, int(0.2 * len(images)))
    embedder = mae.MAEEmbedder(patch_size=PATCH_SIZE, mask_ratio=0.75)
    embedder.fit(images[perm[n_val:]], val_images=images[perm[:n_val]],
                 epochs=epochs, seed=seed)
    return embedder.embed(images)          # (Np, embed_dim), row-major within cutout

In [ ]:
features = {}

if ARMS["raw"]:
    features["raw"] = patches[sweep_idx]

if ARMS["raw_anomaly"]:
    # per-patch, per-channel mean removed -- keeps structure, drops level
    p = patches[sweep_idx].reshape(len(sweep_idx), len(data_channels), PATCH_SIZE * PATCH_SIZE)
    features["raw_anomaly"] = (p - p.mean(axis=2, keepdims=True)).reshape(len(sweep_idx), -1)
    del p

if ARMS["mae"] or ARMS["mae_normpix"]:
    images = dataset.preprocess_for_training()            # (N, C, H, W)
    print("images", images.shape)

if ARMS["mae"]:
    t = time.time()
    emb = train_mae(images)
    assert len(emb) == N_PATCHES, f"MAE token count {len(emb)} != patch count {N_PATCHES}"
    features["mae"] = emb[sweep_idx]
    print(f"mae: {emb.shape} in {time.time() - t:.0f}s")
    del emb; free_gpu()

if ARMS["mae_normpix"]:
    t = time.time()
    with norm_pix_target():
        emb = train_mae(images)
    assert len(emb) == N_PATCHES
    features["mae_normpix"] = emb[sweep_idx]
    print(f"mae_normpix: {emb.shape} in {time.time() - t:.0f}s")
    del emb; free_gpu()

for name, F in features.items():
    print(f"{name:<14} {F.shape}")

## UMAP -- identical parameters for every arm

In [ ]:
embeddings = {}
for name, F in features.items():
    t = time.time()
    s = SingleNemi(params={"device": "gpu", "embedding_dict": dict(UMAP_PARAMS)})
    s.fit_embedding(np.ascontiguousarray(F, dtype="float32"))
    embeddings[name] = np.asarray(s.embedding)
    print(f"{name:<14} -> {embeddings[name].shape}  ({time.time() - t:.0f}s)")
    s = None; free_gpu()

## Diagnostic 1 -- stability

`sampling` is the number that matters: two disjoint subsamples, each labelling the
same eval set.  `seed_only` refits on the *same* subsample with a different KMeans
seed, isolating initialisation noise.  A large gap between them means the
instability is genuinely from resampling the data, not from the clusterer.

In [ ]:
def _labels(fit_X, eval_X, k, seed):
    return np.asarray(KMeans(n_clusters=k, n_init=10, random_state=seed)
                      .fit(fit_X).predict(eval_X))


def stability_curve(embedding, k_values=K_VALUES, n_sub=N_SUB, n_eval=N_EVAL, seed=SEED):
    """ARI between labellings of a common eval set from two disjoint fit subsamples."""
    r = np.random.default_rng(seed)
    pick = r.choice(len(embedding), 2 * n_sub + n_eval, replace=False)
    A, B, E = (embedding[pick[:n_sub]],
               embedding[pick[n_sub:2 * n_sub]],
               embedding[pick[2 * n_sub:]])
    rows = []
    for k in k_values:
        a  = _labels(A, E, k, seed)
        b  = _labels(B, E, k, seed)
        a2 = _labels(A, E, k, seed + 1)                # same data, different seed
        rows.append({"k": k,
                     "sampling":  adjusted_rand_score(a, b),
                     "seed_only": adjusted_rand_score(a, a2)})
        print(f"  k={k:>5}  sampling={rows[-1]['sampling']:.3f}  "
              f"seed_only={rows[-1]['seed_only']:.3f}")
    return pd.DataFrame(rows)


stability = {}
for name, E in embeddings.items():
    print(name)
    stability[name] = stability_curve(E)
    free_gpu()

## Diagnostics 2 and 3 -- neighbour structure

Run on the **feature space** (where the learned metric lives) and on the **UMAP
embedding** (what actually gets clustered).  Comparing the two says whether UMAP
preserves or destroys whatever the representation gained.

In [ ]:
def neighbor_position(space, pos_rc, k=KNN_K, seed=SEED):
    """Mean within-cutout grid distance to k-NN vs random.
    ratio < 1 => neighbours share a grid position -- a position confound, not physics."""
    _, idx = ev.nearest_neighbors(space, k)
    d_nn = np.linalg.norm(pos_rc[:, None, :] - pos_rc[idx], axis=-1).mean()
    rand = np.random.default_rng(seed).integers(0, len(pos_rc), size=idx.shape)
    d_rand = np.linalg.norm(pos_rc[:, None, :] - pos_rc[rand], axis=-1).mean()
    print(f"NN grid dist {d_nn:.2f} | random {d_rand:.2f} | ratio {d_nn / d_rand:.3f}")
    print("(ratio < 1 => neighbours share within-cutout position)")
    return {"nn": float(d_nn), "random": float(d_rand), "ratio": float(d_nn / d_rand)}


knn_sel = np.sort(np.random.default_rng(SEED).choice(len(sweep_idx),
                                                     min(KNN_N, len(sweep_idx)), replace=False))
g_idx  = sweep_idx[knn_sel]                       # back into full-patch indexing
d_k, lon_k, lat_k, pos_k = desc[g_idx], lon[g_idx], lat[g_idx], pos_rc[g_idx]

neighbor = []
for name in embeddings:
    for space_name, space in (("features", features[name][knn_sel]),
                              ("umap", embeddings[name][knn_sel])):
        print(f"\n=== {name} / {space_name} ===")
        S = np.ascontiguousarray(space, dtype="float32")
        con = ev.neighbor_consistency(S, d_k, chan_names, k=KNN_K, seed=SEED)
        geo = ev.neighbor_geographic(S, lon_k, lat_k, k=KNN_K, seed=SEED)
        pos_r = neighbor_position(S, pos_k, k=KNN_K, seed=SEED)
        neighbor.append({"arm": name, "space": space_name,
                         "consistency": con["overall"],
                         "geographic": geo["ratio"],
                         "position": pos_r["ratio"]})
        free_gpu()

neighbor = pd.DataFrame(neighbor)
neighbor

## Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, df in stability.items():
    axes[0].plot(df["k"], df["sampling"], marker="o", label=name)
    axes[0].plot(df["k"], df["seed_only"], marker=".", ls=":", alpha=0.4, color=axes[0].lines[-1].get_color())
axes[0].set_xscale("log"); axes[0].set_xlabel("k"); axes[0].set_ylabel("ARI")
axes[0].set_title("stability (solid = resampling, dotted = seed only)")
axes[0].set_ylim(0, 1); axes[0].legend(); axes[0].grid(alpha=0.3)

piv = neighbor[neighbor.space == "umap"].set_index("arm")[["consistency", "geographic", "position"]]
piv.plot.bar(ax=axes[1], rot=0)
axes[1].axhline(1.0, color="k", lw=1, ls="--")
axes[1].set_ylabel("ratio vs random")
axes[1].set_title("neighbour structure in the UMAP embedding\n(consistency low = good; geographic/position low = confound)")
axes[1].grid(alpha=0.3, axis="y")

fig.tight_layout(); plt.show()

In [ ]:
summary = pd.DataFrame([
    {"arm": name,
     "ARI@20":  float(df.loc[df.k == 20, "sampling"].iloc[0]),
     "ARI@350": float(df.loc[df.k == 350, "sampling"].iloc[0]),
     "k_ARI>0.5": int(df.loc[df["sampling"] > 0.5, "k"].max()) if (df["sampling"] > 0.5).any() else 0}
    for name, df in stability.items()
]).merge(neighbor[neighbor.space == "umap"].drop(columns="space"), on="arm")

display(summary)

with open("embedding_comparison.pkl", "wb") as f:
    pickle.dump({"stability": stability, "neighbor": neighbor, "summary": summary,
                 "config": {"UMAP_PARAMS": UMAP_PARAMS, "K_VALUES": K_VALUES,
                            "SWEEP_N": SWEEP_N, "MAE_EPOCHS": MAE_EPOCHS,
                            "data_channels": data_channels, "PATCH_SIZE": PATCH_SIZE}}, f)
print("wrote embedding_comparison.pkl")

## Reading the output

`k_ARI>0.5` is the headline: the largest k at which a partition still reproduces
across independent draws of the data.

- **MAE's curve extends to larger k than raw** -- the learned metric bought real
  separability.  Cluster at that k, not at the BIC elbow.
- **Curves lie on top of each other** -- the representation is not the binding
  constraint, and the field is a continuum in every metric tried.  Switch to
  ranking on a continuous score rather than partitioning.
- **`raw_anomaly` matches or beats `mae`** -- the patch *level* was the problem and
  a mean subtraction fixes it.  Far cheaper than a training run.
- **`mae_normpix` beats `mae`** -- confirms the loss was spending its capacity on
  level rather than texture; make it the default.
- **`position` ratio well below 1 for an MAE arm** -- the positional embedding is
  leaking. Since gradb2-weighted sampling puts fronts at cutout centres, that can
  masquerade as a front signal.  Treat any structure from that arm as unproven
  until it is retested with position jitter.

`geographic` well below 1 in any arm means that manifold is still largely a map of
the ocean, which is the failure mode this comparison exists to detect.